<h3 style="color: #e0e0ff; font-style: italic;">The Event Adversarial Neural Network (EANN) Model</h3>
<p style="font-size: 0.95em; color: #8a8ab0;"><i>Modernized implementation based on the KDD 2018 paper: <a href="https://arxiv.org/abs/1805.10515" style="color: #a0a0ff; text-decoration: underline;">EANN: Event Adversarial Neural Networks for Multi-Modal Fake News Detection</a></i></p>

---

Most fake news detectors memorize domain-specific terms (e.g., specific politician names or event keywords) instead of learning generalizable features of deception. When the event or domain changes, their performance drops significantly.

**Event Adversarial Neural Network (EANN)** addresses this with a min-max game:
- **Two extractors** (text + image) map inputs to a shared feature space.
- **A fake news detector** reads the fused representation to perform veracity prediction.
- **An event discriminator** tries to predict the event (domain) category of the post — *with its gradients reversed via a Gradient Reversal Layer (GRL).* 

By penalizing the feature extractors when event-specific semantic signatures leak through, EANN forces the representations to be domain-invariant, preserving only the features that signal deception.

**Core Stack and Configuration**

In [6]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# Metrics 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, roc_auc_score, roc_curve, auc
)

# Housekeeping
import warnings
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}")

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

SAVE_DIR = '../../models/eann'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs('../../reports/eann/figures', exist_ok=True)
os.makedirs('../../reports/eann/metrics', exist_ok=True)
sns.set_palette("husl")

sys.path.insert(0, os.path.abspath("../../src"))
from data_pipeline import M4FCDataPipeline

🖥️  Device: cuda


**Dataset Loading & Preprocessing**

In [7]:
print("📂 Loading pre-extracted features and CSV metadata...")

# Load all features
img_feats = torch.load('../../data/features/image_features.pt')
text_feats = torch.load('../../data/features/text_features.pt')
labels = torch.load('../../data/features/labels.pt')
df = pd.read_csv('../../data/M4FC.csv')

# Align all to minimum length
min_len = min(len(img_feats), len(text_feats), len(labels), len(df))
img_feats = img_feats[:min_len]
text_feats = text_feats[:min_len]
labels = labels[:min_len].long()
df = df.iloc[:min_len].reset_index(drop=True)

print(f"📦 Loaded image features: {img_feats.shape}")
print(f"📦 Loaded text features: {text_feats.shape}")
print(f"📋 Metadata DataFrame: {len(df)} samples")
print(f"🧹 Aligned Dataset Size: {min_len} samples")

# Encode event labels
le = LabelEncoder()
event_labels = torch.tensor(le.fit_transform(df['claim_language'].astype(str)), dtype=torch.long)
print(f"🌍 Event classes (domains): {len(le.classes_)} | Classes: {list(le.classes_)}")

📂 Loading pre-extracted features and CSV metadata...
📦 Loaded image features: torch.Size([4948, 512])
📦 Loaded text features: torch.Size([4948, 768])
📋 Metadata DataFrame: 4948 samples
🧹 Aligned Dataset Size: 4948 samples
🌍 Event classes (domains): 10 | Classes: ['Arabic', 'Dutch', 'English', 'French', 'German', 'Portuguese', 'Spanish', 'Tamil', 'Telugu', 'Turkish']


In [8]:
class EANNDataset(Dataset):
    def __init__(self, img_feats, text_feats, labels, event_labels):
        self.img_feats = img_feats
        self.text_feats = text_feats
        self.labels = labels
        self.event_labels = event_labels
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        return (
            self.img_feats[idx],
            self.text_feats[idx],
            self.labels[idx],
            self.event_labels[idx]
        )

**Gradient Reversal Layer & EANN Model Definition**

In [9]:
class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class EANN(nn.Module):
    def __init__(self, img_dim=512, text_dim=768, num_classes=2, num_events=15):
        super().__init__()
        self.text_proj = nn.Linear(text_dim, img_dim)
        combined_dim = img_dim * 2
        
        self.class_classifier = nn.Sequential(
            nn.Linear(combined_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )
        
        self.event_discriminator = nn.Sequential(
            nn.Linear(combined_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_events)
        )
        
    def forward(self, img_feats, text_feats, alpha=1.0):
        proj_text = self.text_proj(text_feats)
        combined = torch.cat([img_feats, proj_text], dim=1)
        
        # Head 1: Veracity classification
        label_output = self.class_classifier(combined)
        
        # Head 2: Event (Domain) classification with GRL
        reverse_feat = GradientReversal.apply(combined, alpha)
        event_output = self.event_discriminator(reverse_feat)
        
        return label_output, event_output, proj_text

**Standardized EANN Trainer**

In [10]:
class EANNTrainer:
    def __init__(self, model, train_loader, val_loader, test_loader, device='cuda', save_name='EANN', early_stopping_patience=10):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.device = device
        self.save_name = save_name
        self.early_stopping_patience = early_stopping_patience
        
        # Standardizing Optimizer & Scheduler
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=20)
        # ---------------------------------------------------------
        # PONDÉRATION DES CLASSES (Cost-Sensitive Learning)
        # ---------------------------------------------------------
        y_train = self.train_loader.dataset.labels
        num_real = (y_train == 0).sum().item()
        num_fake = (y_train == 1).sum().item()
        
        weight_real = 1.0 / num_real
        weight_fake = 1.0 / num_fake
        weights = torch.tensor([weight_real, weight_fake], dtype=torch.float).to(device)
        weights = weights / weights.sum()
        
        self.criterion_label = nn.CrossEntropyLoss(weight=weights)
        # ---------------------------------------------------------
        self.criterion_event = nn.CrossEntropyLoss()
        
        self.metrics_history = {
            'train_loss': [], 'val_loss': [],
            'train_acc': [], 'val_acc': [],
            'precision': [], 'recall': [], 'f1': [],
            'auc': [], 'event_acc': []
        }
        self.best_val_f1 = 0.0
        self.best_val_f1 = 0.0
        self.best_val_acc = 0.0
        self.best_model_path = f"../../models/eann/{save_name}_classifier_best.pth"
        os.makedirs(os.path.dirname(self.best_model_path), exist_ok=True)

    def train_epoch(self, epoch, total_epochs):
        self.model.train()
        total_loss = 0
        total_label_loss = 0
        total_event_loss = 0
        predictions = []
        labels = []
        n_batches = len(self.train_loader)
        
        for batch_idx, (img_feats, text_feats, label, event_label) in enumerate(self.train_loader):
            img_feats = img_feats.to(self.device)
            text_feats = text_feats.to(self.device)
            label = label.to(self.device)
            event_label = event_label.to(self.device)
            
            # Progressive alpha schedule (0 -> 1)
            global_progress = (epoch * n_batches + batch_idx) / (total_epochs * n_batches)
            alpha = 2.0 / (1.0 + np.exp(-10 * global_progress)) - 1.0
            
            self.optimizer.zero_grad()
            label_out, event_out, _ = self.model(img_feats, text_feats, alpha)
            
            loss_label = self.criterion_label(label_out, label)
            loss_event = self.criterion_event(event_out, event_label)
            loss = loss_label + loss_event
            
            loss.backward()
            self.optimizer.step()
            
            total_loss += loss.item()
            total_label_loss += loss_label.item()
            total_event_loss += loss_event.item()
            
            _, preds = torch.max(label_out, 1)
            predictions.extend(preds.cpu().numpy())
            labels.extend(label.cpu().numpy())
            
        self.scheduler.step()
        avg_loss = total_loss / n_batches
        avg_label_loss = total_label_loss / n_batches
        avg_event_loss = total_event_loss / n_batches
        accuracy = accuracy_score(labels, predictions)
        return avg_loss, avg_label_loss, avg_event_loss, accuracy

    @torch.no_grad()
    def evaluate(self, data_loader):
        self.model.eval()
        total_loss = 0
        predictions = []
        probabilities = []
        labels = []
        event_preds = []
        event_labels = []
        
        for img_feats, text_feats, label, event_label in data_loader:
            img_feats = img_feats.to(self.device)
            text_feats = text_feats.to(self.device)
            label = label.to(self.device)
            event_label = event_label.to(self.device)
            
            # Inference uses alpha=0.0 to disable gradient reversal influence
            label_out, event_out, _ = self.model(img_feats, text_feats, alpha=0.0)
            loss_label = self.criterion_label(label_out, label)
            loss_event = self.criterion_event(event_out, event_label)
            loss = loss_label + loss_event
            
            total_loss += loss.item()
            probs = F.softmax(label_out, dim=1)
            _, preds = torch.max(label_out, 1)
            _, ev_preds = torch.max(event_out, 1)
            
            predictions.extend(preds.cpu().numpy())
            probabilities.extend(probs[:, 1].cpu().numpy())
            labels.extend(label.cpu().numpy())
            event_preds.extend(ev_preds.cpu().numpy())
            event_labels.extend(event_label.cpu().numpy())
            
        avg_loss = total_loss / len(data_loader)
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary', zero_division=0)
        event_accuracy = accuracy_score(event_labels, event_preds)
        
        try:
            auc = roc_auc_score(labels, probabilities)
        except Exception:
            auc = 0.5
            
        return {
            'loss': avg_loss,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'auc': auc,
            'event_accuracy': event_accuracy,
            'predictions': predictions,
            'true_labels': labels,
            'probabilities': probabilities,
            'event_predictions': event_preds,
            'event_labels': event_labels
        }

    def train(self, epochs=100):
        print(f"🚀 Starting {self.save_name} Training Pipeline...")
        print("=" * 60)
        patience_counter = 0
        
        for epoch in range(epochs):
            train_loss, train_label_loss, train_event_loss, train_acc = self.train_epoch(epoch, epochs)
            val_metrics = self.evaluate(self.val_loader)
            
            self.metrics_history['train_loss'].append(train_loss)
            self.metrics_history['train_acc'].append(train_acc)
            self.metrics_history['val_loss'].append(val_metrics['loss'])
            self.metrics_history['val_acc'].append(val_metrics['accuracy'])
            self.metrics_history['precision'].append(val_metrics['precision'])
            self.metrics_history['recall'].append(val_metrics['recall'])
            self.metrics_history['f1'].append(val_metrics['f1'])
            self.metrics_history['auc'].append(val_metrics['auc'])
            self.metrics_history['event_acc'].append(val_metrics['event_accuracy'])
            
            print(f"Epoch {epoch+1:2d}/{epochs} | Train Loss: {train_loss:.4f} (Label: {train_label_loss:.4f} | Event: {train_event_loss:.4f}) | Val Acc: {val_metrics['accuracy']:.4f} | Val F1: {val_metrics['f1']:.4f} | Event Acc: {val_metrics['event_accuracy']:.4f}")
            
            if val_metrics['f1'] > self.best_val_f1:
                self.best_val_f1 = val_metrics['f1']
                self.best_val_acc = val_metrics['accuracy']
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'val_accuracy': val_metrics['accuracy']
                }, self.best_model_path)
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= self.early_stopping_patience:
                    print(f"🛑 Early stopping triggered at epoch {epoch+1}!")
                    break
                    
        return self.metrics_history

    def test(self):
        print(f"Evaluating best {self.save_name} model on test set...")
        if os.path.exists(self.best_model_path):
            checkpoint = torch.load(self.best_model_path)
            self.model.load_state_dict(checkpoint['model_state_dict'])
        test_metrics = self.evaluate(self.test_loader)
        print(f"Test Accuracy: {test_metrics['accuracy']:.4f} | Test F1: {test_metrics['f1']:.4f} | Test AUC: {test_metrics['auc']:.4f} | Test Event Acc: {test_metrics['event_accuracy']:.4f}")
        return test_metrics


**Standardized EANN Visualizer**

In [11]:
class EANNVisualizer:
    @staticmethod
    def plot_all(metrics_history, test_metrics, save_name):
        fig_dir = f"../../reports/eann/figures/{save_name}"
        os.makedirs(fig_dir, exist_ok=True)
        
        # 1. Training History
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes[0, 0].plot(metrics_history['train_loss'], label='Train Loss')
        axes[0, 0].plot(metrics_history['val_loss'], label='Val Loss')
        axes[0, 0].set_title('Loss Over Time')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        axes[0, 1].plot(metrics_history['train_acc'], label='Train Acc')
        axes[0, 1].plot(metrics_history['val_acc'], label='Val Acc')
        axes[0, 1].set_title('Accuracy Over Time')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Accuracy')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        axes[1, 0].plot(metrics_history['f1'], label='F1-Score', marker='o')
        axes[1, 0].plot(metrics_history['auc'], label='AUC', marker='s')
        axes[1, 0].set_title('F1-Score and AUC')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Score')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        axes[1, 1].plot(metrics_history['precision'], label='Precision', marker='^')
        axes[1, 1].plot(metrics_history['recall'], label='Recall', marker='v')
        axes[1, 1].set_title('Precision-Recall Trade-off')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Score')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(fig_dir, 'loss_accuracy_history.png'), dpi=100)
        plt.close()
        
        # 2. Confusion Matrix
        cm = confusion_matrix(test_metrics['true_labels'], test_metrics['predictions'])
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
        plt.title(f'Confusion Matrix - {save_name}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.savefig(os.path.join(fig_dir, 'confusion_matrix.png'), dpi=100)
        plt.close()
        
        # 3. ROC Curve
        fpr, tpr, _ = roc_curve(test_metrics['true_labels'], test_metrics['probabilities'])
        roc_auc = auc(fpr, tpr)
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, color='green', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve - {save_name}')
        plt.legend(loc="lower right")
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(fig_dir, 'roc_curve.png'), dpi=100)
        plt.close()
        
        # 4. Prediction Distribution
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].hist(test_metrics['probabilities'], bins=30, edgecolor='black', alpha=0.7, color='green')
        axes[0].axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Decision Boundary')
        axes[0].set_title('Prediction Probability Distribution')
        axes[0].set_xlabel('Probability of Fake')
        axes[0].set_ylabel('Frequency')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        axes[1].boxplot(test_metrics['probabilities'], vert=False, widths=0.5)
        axes[1].scatter(test_metrics['probabilities'], [1] * len(test_metrics['probabilities']), alpha=0.1, color='green')
        axes[1].axvline(x=0.5, color='red', linestyle='--', linewidth=2)
        axes[1].set_title('Prediction Probability Box Plot')
        axes[1].set_xlabel('Probability of Fake')
        
        plt.tight_layout()
        plt.savefig(os.path.join(fig_dir, 'prediction_distribution.png'), dpi=100)
        plt.close()
        print(f"✅ All plots saved to {fig_dir}")

**Standardized EANN Model Saver & Reporter**

In [12]:
class EANNModelSaver:
    @staticmethod
    def generate_report(metrics_history, test_metrics, save_name):
        report_path = f"../../reports/eann/metrics/{save_name}_Report.txt"
        os.makedirs(os.path.dirname(report_path), exist_ok=True)
        
        with open(report_path, 'w') as f:
            f.write("="*60 + "\n")
            f.write(f"{save_name} MODEL FAKE NEWS DETECTION REPORT\n")
            f.write("="*60 + "\n\n")
            
            f.write("1. PERFORMANCE METRICS (TEST SET)\n")
            f.write("-"*35 + "\n")
            f.write(f"Accuracy:        {test_metrics['accuracy']:.4f}\n")
            f.write(f"Precision:       {test_metrics['precision']:.4f}\n")
            f.write(f"Recall:          {test_metrics['recall']:.4f}\n")
            f.write(f"F1 Score:        {test_metrics['f1']:.4f}\n")
            f.write(f"ROC-AUC:         {test_metrics['auc']:.4f}\n")
            f.write(f"Event Accuracy:  {test_metrics['event_accuracy']:.4f}\n\n")
            
            f.write("2. CONFUSION MATRIX\n")
            f.write("-"*35 + "\n")
            cm = confusion_matrix(test_metrics['true_labels'], test_metrics['predictions'])
            f.write(f"True Real, Predicted Real: {cm[0,0]}\n")
            f.write(f"True Real, Predicted Fake: {cm[0,1]}\n")
            f.write(f"True Fake, Predicted Real: {cm[1,0]}\n")
            f.write(f"True Fake, Predicted Fake: {cm[1,1]}\n\n")
            
            f.write("3. CONVERGENCE HISTORY\n")
            f.write("-"*35 + "\n")
            f.write(f"Best Val Accuracy: {max(metrics_history['val_acc']):.4f}\n")
            f.write(f"Final Train Accuracy: {metrics_history['train_acc'][-1]:.4f}\n")
            f.write(f"Final Val Accuracy:   {metrics_history['val_acc'][-1]:.4f}\n")
            f.write(f"Final Train Loss:     {metrics_history['train_loss'][-1]:.4f}\n")
            f.write(f"Final Val Loss:       {metrics_history['val_loss'][-1]:.4f}\n")
            
        print(f"✅ Performance report saved to {report_path}")

**Standard Split Performance Benchmarking (70/15/15 split)**

In [13]:
# ── Pipeline standardisée : split stratifié (70/15/15) ──
pipeline = M4FCDataPipeline(
    csv_path     = "../../data/M4FC.csv",
    target_col   = "target",
    random_state = 42,
    normalize    = False, # Les features pré-extraites n'ont pas besoin de scaler
)

tensors = pipeline.split_and_resample_tensors(img_feats, text_feats, labels)

img_train, txt_train, y_train = tensors["img_train"], tensors["txt_train"], tensors["y_train"]
img_val, txt_val, y_val = tensors["img_val"], tensors["txt_val"], tensors["y_val"]
img_test, txt_test, y_test = tensors["img_test"], tensors["txt_test"], tensors["y_test"]

# Extract indices to align event_labels
indices = np.arange(min_len)
idx_temp, idx_test = train_test_split(
    indices,
    test_size=pipeline.test_size,
    random_state=pipeline.random_state,
    stratify=labels.numpy(),
)
relative_val_size = pipeline.val_size / (1.0 - pipeline.test_size)
idx_train, idx_val = train_test_split(
    idx_temp,
    test_size=relative_val_size,
    random_state=pipeline.random_state,
    stratify=labels[idx_temp].numpy(),
)

event_train = event_labels[idx_train]
event_val = event_labels[idx_val]
event_test = event_labels[idx_test]

print("🔄 Initializing Dataset and splits matching HMCAN seed 42...")
print(f"   Train size: {len(y_train):,}")
print(f"   Val size:   {len(y_val):,}")
print(f"   Test size:  {len(y_test):,}")

# Datasets
train_dataset = EANNDataset(img_train, txt_train, y_train, event_train)
val_dataset = EANNDataset(img_val, txt_val, y_val, event_val)
test_dataset = EANNDataset(img_test, txt_test, y_test, event_test)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Model configuration
num_events = len(np.unique(event_labels.numpy()))
model = EANN(img_dim=512, text_dim=768, num_classes=2, num_events=num_events)

# Initialize trainer
trainer = EANNTrainer(model, train_loader, val_loader, test_loader, device=device, save_name="EANN_Standard")

# Train EANN
metrics_history = trainer.train(epochs=100)

# Evaluate on test set
test_metrics = trainer.test()

# Plot results
EANNVisualizer.plot_all(metrics_history, test_metrics, save_name="eann")

# Generate report
EANNModelSaver.generate_report(metrics_history, test_metrics, save_name="EANN_Standard")

✂️  Tensor split completed → train=3,462 | val=743 | test=743
🔄 Initializing Dataset and splits matching HMCAN seed 42...
   Train size: 3,462
   Val size:   743
   Test size:  743
🚀 Starting EANN_Standard Training Pipeline...
Epoch  1/100 | Train Loss: 2.1739 (Label: 0.5836 | Event: 1.5903) | Val Acc: 0.6703 | Val F1: 0.7882 | Event Acc: 0.8075
Epoch  2/100 | Train Loss: 1.2805 (Label: 0.4253 | Event: 0.8552) | Val Acc: 0.7685 | Val F1: 0.8608 | Event Acc: 0.8102
Epoch  3/100 | Train Loss: 1.1027 (Label: 0.3530 | Event: 0.7497) | Val Acc: 0.8156 | Val F1: 0.8922 | Event Acc: 0.7914
Epoch  4/100 | Train Loss: 0.9799 (Label: 0.3159 | Event: 0.6640) | Val Acc: 0.8546 | Val F1: 0.9173 | Event Acc: 0.8237
Epoch  5/100 | Train Loss: 0.7520 (Label: 0.2749 | Event: 0.4772) | Val Acc: 0.8829 | Val F1: 0.9347 | Event Acc: 0.8385
Epoch  6/100 | Train Loss: 0.7163 (Label: 0.2454 | Event: 0.4709) | Val Acc: 0.9233 | Val F1: 0.9584 | Event Acc: 0.8048
Epoch  7/100 | Train Loss: 0.7170 (Label: 0.214

**Model Registry Registration**

In [14]:
# Update model registry to track the newly trained EANN champion
sys.path.insert(0, os.path.abspath("../../src"))
from model_registry import update_registry

update_registry(
    name="EANN",
    arch="EANNWithFeatures",
    rel_path="models/eann/EANN_Standard_classifier_best.pth",
    val_accuracy=max(metrics_history['val_acc']),
    val_f1=metrics_history['f1'][np.argmax(metrics_history['val_acc'])],
    test_accuracy=test_metrics['accuracy'],
    test_f1=test_metrics['f1'],
    test_auc=test_metrics['auc'],
    img_dim=512,
    text_dim=768,
    num_classes=2,
    num_events=num_events
)

ℹ️  Registry unchanged — current champion 'Fusion - Metadata Fusion (MLP)' (acc=0.9462) leads 'EANN' (acc=0.9394).


False

**Zero-Shot Domain Generalization Test**

In [15]:
# Zero-Shot split based on domains (languages)
unseen_domains = ['Spanish', 'Arabic']
seen_domains = [d for d in le.classes_ if d not in unseen_domains]
print(f"👁️  Seen events (domains used for training): {seen_domains}")
print(f"🚫 Unseen events (domains held out for testing): {unseen_domains}")

df_aligned = df.iloc[:min_len].reset_index(drop=True)
seen_mask = ~df_aligned['claim_language'].isin(unseen_domains)
unseen_mask = df_aligned['claim_language'].isin(unseen_domains)

# Features for seen domains
img_seen = img_feats[seen_mask]
txt_seen = text_feats[seen_mask]
y_seen = labels[seen_mask]
event_seen = event_labels[seen_mask]

# Features for unseen domains (testing only)
img_unseen = img_feats[unseen_mask]
txt_unseen = text_feats[unseen_mask]
y_unseen = labels[unseen_mask]
event_unseen = event_labels[unseen_mask]

print("📊 Split Sizes:")
print(f"   Seen domains sample count:   {len(y_seen):,}")
print(f"   Unseen domains sample count: {len(y_unseen):,}")

# Split the seen domains into train and val (85/15)
indices_seen = np.arange(len(y_seen))
idx_train, idx_val = train_test_split(
    indices_seen,
    test_size=0.15,
    random_state=42,
    stratify=y_seen.numpy(),
)

# Construct splits
img_train, txt_train, y_train = img_seen[idx_train], txt_seen[idx_train], y_seen[idx_train]
img_val, txt_val, y_val = img_seen[idx_val], txt_seen[idx_val], y_seen[idx_val]
img_test, txt_test, y_test = img_unseen, txt_unseen, y_unseen

# Map seen event labels to a continuous range 0..X safely using numpy arrays
seen_languages = df_aligned['claim_language'].iloc[seen_mask].values
languages_train = seen_languages[idx_train]
languages_val = seen_languages[idx_val]

le_seen = LabelEncoder()
event_train_mapped = torch.tensor(le_seen.fit_transform(languages_train), dtype=torch.long)
event_val_mapped = torch.tensor(le_seen.transform(languages_val), dtype=torch.long)
event_test_mapped = torch.zeros_like(y_test)  # Dummy event labels for testing

# Datasets
train_dataset = EANNDataset(img_train, txt_train, y_train, event_train_mapped)
val_dataset = EANNDataset(img_val, txt_val, y_val, event_val_mapped)
test_dataset = EANNDataset(img_test, txt_test, y_test, event_test_mapped)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

num_events_seen = len(le_seen.classes_)
model_zs = EANN(img_dim=512, text_dim=768, num_classes=2, num_events=num_events_seen)

# Initialize trainer
trainer_zs = EANNTrainer(model_zs, train_loader, val_loader, test_loader, device=device, save_name="EANN_ZeroShot")

# Train EANN ZeroShot
metrics_history_zs = trainer_zs.train(epochs=100)

# Evaluate on test set
test_metrics_zs = trainer_zs.test()

# True Zero-Shot Generalization Results
seen_val_acc = max(metrics_history_zs['val_acc'])
unseen_test_acc = test_metrics_zs['accuracy']
gen_gap = (seen_val_acc - unseen_test_acc) * 100

print("\n" + "─" * 50)
print("📊 True Zero-Shot Generalization Results")
print("─" * 50)
print(f"   👁️  Seen domains (Val):   {seen_val_acc:.4f}  ({seen_val_acc*100:.2f}%)")
print(f"   🚫 Unseen domains (Test): {unseen_test_acc:.4f}  ({unseen_test_acc*100:.2f}%)")
print(f"   📉 Domain generalization gap: {gen_gap:.2f}%")
print("─" * 50)
if gen_gap > 10:
    print("   ⚠️  Large gap — Domain-invariant learning is insufficient.")
elif gen_gap > 5:
    print("   ⚠️  Moderate gap — Subtle domain indicators still leak.")
else:
    print("   ✅ Robust generalization — Domain-invariant features successfully learned.")

# Plot and save
EANNVisualizer.plot_all(metrics_history_zs, test_metrics_zs, save_name="eann_zeroshot")

# Generate report
EANNModelSaver.generate_report(metrics_history_zs, test_metrics_zs, save_name="EANN_ZeroShot")

👁️  Seen events (domains used for training): ['Dutch', 'English', 'French', 'German', 'Portuguese', 'Tamil', 'Telugu', 'Turkish']
🚫 Unseen events (domains held out for testing): ['Spanish', 'Arabic']
📊 Split Sizes:
   Seen domains sample count:   4,140
   Unseen domains sample count: 808
🚀 Starting EANN_ZeroShot Training Pipeline...
Epoch  1/100 | Train Loss: 1.9295 (Label: 0.5479 | Event: 1.3816) | Val Acc: 0.7262 | Val F1: 0.8283 | Event Acc: 0.8712
Epoch  2/100 | Train Loss: 1.1489 (Label: 0.4152 | Event: 0.7338) | Val Acc: 0.7794 | Val F1: 0.8671 | Event Acc: 0.8615
Epoch  3/100 | Train Loss: 0.9803 (Label: 0.3518 | Event: 0.6285) | Val Acc: 0.8035 | Val F1: 0.8831 | Event Acc: 0.8196
Epoch  4/100 | Train Loss: 0.7769 (Label: 0.3093 | Event: 0.4676) | Val Acc: 0.8245 | Val F1: 0.8980 | Event Acc: 0.8873
Epoch  5/100 | Train Loss: 0.6310 (Label: 0.2758 | Event: 0.3553) | Val Acc: 0.8728 | Val F1: 0.9282 | Event Acc: 0.8663
Epoch  6/100 | Train Loss: 0.6647 (Label: 0.2425 | Event: 0.